# 06.02 链表与连续数组访存对比实验

本 Notebook 是本章核心实验入口。它以桶化 Cuckoo Hash（布谷鸟哈希，一种为每个键设置少量候选位置的哈希表方法）查询作为连续数组访存案例，并对照链地址法的链式访问问题。请按顺序执行：**工作区 → 环境 → CPU（Central Processing Unit，中央处理器）数据布局 → 链式/连续访存对比 → Tiling（数据切分） → Kernel（NPU 核函数） → Host（主机侧程序） → 构建 → NPU（Neural Processing Unit，神经网络处理器）运行 → 精确验证**。

## 学习目标

理解链表指针跳转、连续数组分块搬运、链地址法冲突处理和桶化连续数组布局；了解 GM（Global Memory，全局内存）、UB（Unified Buffer，统一缓冲区）、AI Core。

完成后能够解释链表式访问为什么难以稳定分块，说明连续桶化数组如何把每个 query（查询请求）的候选访问固定为两个 32B 桶，并独立构建可验证的连续数组查找工程。

## 0. 创建 Notebook 工作区

`WORK_DIR` 是运行时工程目录。后续 `%%writefile` 单元会把 C++/Ascend C/Python 工程文件写入这里。为保持与 05 章实验一致，本实验默认使用课程根目录下的 `work/06.02_memory_access_compare`，避免把编译工程放在中文章节目录内。

如需手动指定工作目录，可在运行本单元前设置环境变量 `MEM_ACCESS_WORK_DIR`。删除工作目录不会损坏课程，只需重新顺序执行所有 `%%writefile` 单元即可恢复。

In [ ]:
from pathlib import Path

import os
import shutil
import subprocess
import sys

# 本单元只负责定位并创建工作区，不生成任何源码。
# 后续所有 %%writefile 都依赖 WRITE_ROOT，因此必须先运行本单元。

chapter_name = "06_linked_list_vs_contiguous_array"
cwd = Path.cwd().resolve()
if (cwd / "06.02_memory_access_compare.ipynb").exists():
    # 情况一：从第六章目录启动 notebook。
    CHAPTER_DIR = cwd
elif (cwd / chapter_name / "06.02_memory_access_compare.ipynb").exists():
    # 情况二：从课程根目录启动 notebook。
    CHAPTER_DIR = cwd / chapter_name
else:
    raise FileNotFoundError("请从课程根目录或 06 章节目录启动 Notebook")
COURSE_DIR = CHAPTER_DIR.parent

env_work_dir = os.environ.get("MEM_ACCESS_WORK_DIR")
if env_work_dir:
    WORK_DIR = Path(env_work_dir).expanduser().resolve()
else:
    WORK_DIR = (COURSE_DIR / "work" / "06.02_memory_access_compare").resolve()

# IPython 的 %%writefile 支持变量展开；统一使用 POSIX 风格路径，避免反斜杠转义问题。
WRITE_ROOT = WORK_DIR.as_posix()

print("创建实验工作区")
for relative in ("op_kernel", "op_host", "scripts", "build"):
    # 这些目录分别保存 Kernel、Host、Python 脚本和 CMake 构建产物。
    (WORK_DIR / relative).mkdir(parents=True, exist_ok=True)
print("章节目录：", CHAPTER_DIR)
print("工作区：", WORK_DIR)
print("写入根路径：", WRITE_ROOT)
print("状态：后续 %%writefile 单元会在这里生成完整工程。")

## 1. 检查 CANNLab 环境

`ASCEND_HOME_PATH` 指向 CANN Toolkit（CANN 工具包），`ASCEND_OPP_PATH` 指向算子包，`npu-smi info` 用于查看 NPU。

In [ ]:
# 这些环境变量来自 CANN Toolkit；没有它们时仍可运行 CPU 教学单元。
ASCEND_HOME = os.environ.get("ASCEND_HOME_PATH")
ASCEND_OPP = os.environ.get("ASCEND_OPP_PATH")
NPU_SMI = shutil.which("npu-smi")

print("CANN Toolkit NPU")
print("ASCEND_HOME_PATH =", ASCEND_HOME or "<not set>")
print("ASCEND_OPP_PATH  =", ASCEND_OPP or "<not set>")
print("npu-smi          =", NPU_SMI or "<not found>")
if NPU_SMI:
    # npu-smi 能看到设备，说明至少可以尝试后续 NPU 编译和运行。
    subprocess.run([NPU_SMI, "info"], check=False)

# NPU_READY 控制后续上板单元是否执行；CPU 建表和 Golden 不受影响。
NPU_READY = bool(ASCEND_HOME and NPU_SMI)
print("NPU_READY =", NPU_READY)

## 2. 明确算子接口

本节的 NPU 侧实现采用连续桶化数组查找：`B` 是每张表的桶数且必须为二次幂，`Q` 是 query 数。一个桶为 8 个 `int32`，下标 `bucket*8+slot*2` 保存 key（键，查找时用于定位数据的标识），后一项保存 value（值，与 key 对应的数据）。链表/链地址法作为对照模型用于理解不规则访存，实际 Kernel 选择更适合分块搬运的 packed table（打包后的连续表）。

<table
  align="left"
  style="width: 90%;
         max-width: 1200px;
         margin: 0 auto 0 0 !important;
         text-align: left;">
  <thead><tr>
    <th style="text-align: left;">参数</th>
    <th style="text-align: left;">类型与形状</th>
    <th style="text-align: left;">读写</th>
    <th style="text-align: left;">作用</th>
  </tr></thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><code>table0</code></td>
      <td style="text-align: left;"><code>int32[B*8]</code></td>
      <td style="text-align: left;">只读</td>
      <td style="text-align: left;">第一张连续桶化表</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>table1</code></td>
      <td style="text-align: left;"><code>int32[B*8]</code></td>
      <td style="text-align: left;">只读</td>
      <td style="text-align: left;">第二张连续桶化表</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>queries</code></td>
      <td style="text-align: left;"><code>int32[Q]</code></td>
      <td style="text-align: left;">只读</td>
      <td style="text-align: left;">批量查询 key，按连续块搬运</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>values_out</code></td>
      <td style="text-align: left;"><code>int32[Q]</code></td>
      <td style="text-align: left;">只写</td>
      <td style="text-align: left;">命中 value，未命中填 0</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>found_out</code></td>
      <td style="text-align: left;"><code>int32[Q]</code></td>
      <td style="text-align: left;">只写</td>
      <td style="text-align: left;">命中标记 1/0，避免 value 歧义</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>tiling</code></td>
      <td style="text-align: left;">结构体</td>
      <td style="text-align: left;">只读</td>
      <td style="text-align: left;">桶数、查询数和分核信息</td>
    </tr>
  </tbody>
</table>
<div style="clear: both;"></div>

一次 Kernel launch（核函数启动）完成全部查询。未命中时 `values_out=0`、`found_out=0`；命中 value 即使为 `0` 或负数，也由 `found_out=1`（命中标记为 1）明确标识。

## 3. 编写 CPU 数据布局与 Golden（参考结果）工具

该文件是数据语义的唯一来源，负责生成本实验使用的桶化连续数组，并把它作为链式冲突处理的对照对象：

- `mix32/hash1/hash2` 体现哈希查找：把 key 映射到两个候选桶；
- `build_cuckoo` 体现连续桶化表构造：每个桶固定 4 个槽，底层是连续数组；
- `lookup_cpu` 体现连续数组查找：每个 query 固定检查两张表中的两个桶；
- `validate_packed_tables` 体现工程边界：检查桶数、长度、空槽和 key 是否在正确桶中；
- CPU 双桶查询结果作为 Golden。

先保证 CPU 数据正确，再进入 Host/Kernel 分界。

In [ ]:
%%writefile $WRITE_ROOT/scripts/cuckoo_utils.py
from __future__ import annotations

import math
from dataclasses import dataclass
from typing import Iterable

import numpy as np

# ===== 基本常量：描述连续桶化数组的物理布局 =====
EMPTY_KEY = -1
BUCKET_SIZE = 4
ENTRY_WORDS = 2
BUCKET_WORDS = BUCKET_SIZE * ENTRY_WORDS
MAX_KICKS = 256
MAX_INT32 = np.iinfo(np.int32).max
MIN_INT32 = np.iinfo(np.int32).min
U32_MASK = 0xFFFFFFFF


# CPU 构表结果：除了两张连续表，也记录扩容和踢出次数，便于教学观察。
@dataclass
class BuildResult:
    table0: np.ndarray
    table1: np.ndarray
    bucket_count: int
    resize_count: int
    max_kicks_used: int


def mix32(value: int) -> int:
    """哈希混合函数：把输入 key 打散，降低集中碰撞概率。"""
    # Python 整数不会自然溢出，因此每一步都用 U32_MASK 模拟 C++ uint32_t。
    value &= U32_MASK
    value ^= value >> 16
    value = (value * 0x7FEB352D) & U32_MASK
    value ^= value >> 15
    value = (value * 0x846CA68B) & U32_MASK
    value ^= value >> 16
    return value & U32_MASK


def hash1(key: int, bucket_count: int) -> int:
    # 第一个哈希函数：为 key 计算 table0 中的候选桶号。
    return mix32(key ^ 0x243F6A88) & (bucket_count - 1)


def hash2(key: int, bucket_count: int) -> int:
    # 第二个哈希函数：为 key 计算 table1 中的候选桶号。
    return mix32(key ^ 0x9E3779B9) & (bucket_count - 1)


# 桶数取二次幂后，hash & (bucket_count - 1) 等价于取模，Kernel 中计算更简单。
def next_power_of_two(value: int) -> int:
    result = 1
    while result < value:
        result <<= 1
    return result


def validate_items(items: Iterable[tuple[int, int]]) -> list[tuple[int, int]]:
    # 先在 CPU 侧拒绝非法 key/value，避免 NPU 侧语义不清。
    result = []
    seen = set()
    for key, value in items:
        key, value = int(key), int(value)
        if key < 0 or key > MAX_INT32:
            raise ValueError("key must be a nonnegative int32")
        if value < MIN_INT32 or value > MAX_INT32:
            raise ValueError("value must fit in int32")
        if key in seen:
            raise ValueError("duplicate keys are not allowed")
        seen.add(key)
        result.append((key, value))
    return result


def _empty_table(bucket_count: int) -> np.ndarray:
    # 连续数组布局：[bucket, slot, key/value]，最终会 reshape 成一维 packed table。
    table = np.zeros((bucket_count, BUCKET_SIZE, ENTRY_WORDS), dtype=np.int32)
    table[:, :, 0] = EMPTY_KEY
    return table


def _try_build(items: list[tuple[int, int]], bucket_count: int):
    # Cuckoo 构表阶段在 CPU 完成：冲突时踢出旧元素，再切换到另一张表尝试放置。
    table0 = _empty_table(bucket_count)
    table1 = _empty_table(bucket_count)
    max_kicks_used = 0
    for original_key, original_value in items:
        current_key, current_value = original_key, original_value
        table_id = 0
        placed = False
        for kick in range(MAX_KICKS):
            bucket = (
                hash1(current_key, bucket_count)
                if table_id == 0
                else hash2(current_key, bucket_count)
            )
            table = table0 if table_id == 0 else table1
            # 连续桶化体现：同一个桶内 4 个槽相邻存放，可一次性连续读取。
            empty_slots = np.flatnonzero(table[bucket, :, 0] == EMPTY_KEY)
            if empty_slots.size:
                slot = int(empty_slots[0])
                table[bucket, slot] = (current_key, current_value)
                max_kicks_used = max(max_kicks_used, kick)
                placed = True
                break

            # 桶已满时确定性地选择一个槽踢出，随后转到另一张表。
            slot = mix32(current_key ^ kick) & (BUCKET_SIZE - 1)
            evicted_key = int(table[bucket, slot, 0])
            evicted_value = int(table[bucket, slot, 1])
            table[bucket, slot] = (current_key, current_value)
            current_key, current_value = evicted_key, evicted_value
            table_id = 1 - table_id
        if not placed:
            return None
    return table0.reshape(-1), table1.reshape(-1), max_kicks_used


def build_cuckoo(items: Iterable[tuple[int, int]], load_factor: float = 0.75) -> BuildResult:
    # 若踢出形成循环，则扩大桶数并重新建表；NPU 只处理已经稳定的只读表。
    clean_items = validate_items(items)
    if not 0.0 < load_factor <= 0.75:
        raise ValueError("load_factor must be in (0, 0.75]")
    capacity_per_bucket_pair = 2 * BUCKET_SIZE
    required = max(
        1,
        math.ceil(len(clean_items) / (capacity_per_bucket_pair * load_factor)),
    )
    bucket_count = next_power_of_two(required)
    resize_count = 0
    # 若踢出形成循环，则扩大桶数并重新建表；NPU 只处理已经稳定的只读表。
    while True:
        built = _try_build(clean_items, bucket_count)
        if built is not None:
            table0, table1, max_kicks_used = built
            return BuildResult(
                table0, table1, bucket_count, resize_count, max_kicks_used
            )
        bucket_count <<= 1
        resize_count += 1


def validate_packed_tables(table0, table1, bucket_count: int):
    if bucket_count <= 0 or bucket_count & (bucket_count - 1):
        raise ValueError("bucket_count must be a positive power of two")
    expected = bucket_count * BUCKET_WORDS
    table0 = np.asarray(table0, dtype=np.int32).reshape(-1)
    table1 = np.asarray(table1, dtype=np.int32).reshape(-1)
    if table0.size != expected or table1.size != expected:
        raise ValueError("packed table length mismatch")
    seen = set()
    # 逐桶检查 packed table，确保每个有效 key 都在自己的候选桶中。
    for table_id, flat in enumerate((table0, table1)):
        rows = flat.reshape(bucket_count, BUCKET_SIZE, ENTRY_WORDS)
        for bucket in range(bucket_count):
            for slot in range(BUCKET_SIZE):
                key, value = map(int, rows[bucket, slot])
                if key == EMPTY_KEY:
                    if value != 0:
                        raise ValueError("empty slots must store value 0")
                    continue
                if key < 0 or key in seen:
                    raise ValueError("invalid or duplicate packed key")
                expected_bucket = (
                    hash1(key, bucket_count)
                    if table_id == 0
                    else hash2(key, bucket_count)
                )
                if bucket != expected_bucket:
                    raise ValueError("key is stored in an invalid bucket")
                seen.add(key)


def lookup_cpu(table0, table1, bucket_count: int, queries):
    validate_packed_tables(table0, table1, bucket_count)
    queries = np.asarray(queries, dtype=np.int64).reshape(-1)
    if np.any(queries < 0) or np.any(queries > MAX_INT32):
        raise ValueError("queries must contain nonnegative int32 keys")
    rows0 = np.asarray(table0, np.int32).reshape(bucket_count, BUCKET_SIZE, ENTRY_WORDS)
    rows1 = np.asarray(table1, np.int32).reshape(bucket_count, BUCKET_SIZE, ENTRY_WORDS)
    # values/found 分开输出：value 可以合法等于 0 或 -1，不能用特殊值表示未命中。
    values = np.zeros(queries.size, dtype=np.int32)
    found = np.zeros(queries.size, dtype=np.int32)
    for index, key64 in enumerate(queries):
        key = int(key64)
        # 与链地址法不同，这里不沿 next 指针走链；只扫描两个固定连续桶。
        for table, bucket in (
            (rows0, hash1(key, bucket_count)),
            (rows1, hash2(key, bucket_count)),
        ):
            for slot in range(BUCKET_SIZE):
                if int(table[bucket, slot, 0]) == key:
                    values[index] = table[bucket, slot, 1]
                    found[index] = 1
    return values, found

### 3.1 编写测试数据生成器

`gen_data.py` 将逻辑键值对构造成两张扁平表，并输出 query 与 Golden。固定随机种子确保问题可以复现。`meta.txt` 保存 `bucket_count query_count item_count`，供 Host 参数和日志使用。

In [ ]:
%%writefile $WRITE_ROOT/scripts/gen_data.py
from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np

from cuckoo_utils import build_cuckoo, hash1, hash2, lookup_cpu


def _collision_keys(count: int):
    # 构造集中碰撞案例，用于观察哈希冲突被桶化结构吸收后的行为。
    keys = []
    candidate = 0
    while len(keys) < count:
        if hash1(candidate, 4) == 0 and hash2(candidate, 4) == 0:
            keys.append(candidate)
        candidate += 1
    return keys


def make_case(name: str):
    # 每个 case 覆盖一种教学边界：空表、命中、未命中、碰撞、尾块和随机装载。
    # 空表：验证未命中路径和 found=0。
    if name == "empty":
        return [], [0, 1, 99]
    if name == "single":
        return [(7, -17)], [7]
    # 全命中：验证每个 query 都能在两个候选桶之一找到。
    if name == "hits":
        items = [(key, key * 10) for key in range(1, 17)]
        return items, [key for key, _ in items]
    if name == "misses":
        return [(key, key + 100) for key in range(20)], list(range(100, 117))
    if name == "mixed":
        return [(3, 30), (7, 70), (11, 110), (19, 190)], [3, 4, 19, 20, 7]
    if name == "duplicate_queries":
        return [(5, 50), (9, 90)], [5, 5, 9, 8, 5]
    if name == "negative_values":
        return [(1, -1), (2, -2147483648), (3, 2147483647)], [1, 2, 3, 4]
    # 碰撞案例：多个 key 被设计到相同候选桶，观察桶满、踢出和扩容。
    if name == "collision":
        keys = _collision_keys(12)
        return [(key, 1000 + i) for i, key in enumerate(keys)], keys + [999999]
    # 尾块案例：255/256/257 用来覆盖 QUERY_TILE 边界前后。
    if name in {"tail255", "tail256", "tail257"}:
        query_count = int(name.removeprefix("tail"))
        items = [(key, key * 3 - 5) for key in range(180)]
        queries = [i % 240 for i in range(query_count)]
        return items, queries
    if name in {"random_low", "random_high"}:
        seed = 20260725 if name == "random_low" else 20260726
        rng = np.random.default_rng(seed)
        item_count = 128 if name == "random_low" else 1200
        keys = rng.choice(2_000_000, size=item_count, replace=False).astype(np.int64)
        values = rng.integers(
            np.iinfo(np.int32).min,
            np.iinfo(np.int32).max,
            size=item_count,
            dtype=np.int32,
        )
        items = [(int(key), int(value)) for key, value in zip(keys, values)]
        hit_queries = keys[: item_count // 2]
        miss_queries = np.arange(3_000_000, 3_000_000 + item_count // 2)
        queries = np.concatenate((hit_queries, miss_queries))
        rng.shuffle(queries)
        return items, queries.astype(np.int64).tolist()
    items = [(3, 300), (7, 700), (11, -1100), (18, 1800), (29, 2900)]
    return items, [7, 8, 3, 29, 100, 11]


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--case", default="demo")
    args = parser.parse_args()
    items, queries = make_case(args.case)
    # CPU 建表生成连续 packed table；NPU 侧只做只读查询。
    result = build_cuckoo(items)
    query_array = np.asarray(queries, dtype=np.int32)
    # Golden 与 Kernel 采用同一套双桶查找语义，用于后续精确比对。
    golden_values, golden_found = lookup_cpu(
        result.table0, result.table1, result.bucket_count, query_array
    )

    # 写入二进制文件：Host 和 Kernel 只读取这些连续数组，不再依赖 Python 对象。
    Path("input").mkdir(exist_ok=True)
    Path("output").mkdir(exist_ok=True)
    # table0/table1 是连续桶化数组；queries 是连续 query 数组。
    result.table0.tofile("input/table0.bin")
    result.table1.tofile("input/table1.bin")
    query_array.tofile("input/queries.bin")
    golden_values.tofile("output/golden_values.bin")
    golden_found.tofile("output/golden_found.bin")
    # meta.txt 只保存运行所需的三个标量，供 Host 命令行参数和验证脚本使用。
    Path("meta.txt").write_text(
        f"{result.bucket_count} {query_array.size} {len(items)}\n",
        encoding="utf-8",
    )
    print(
        f"case={args.case}, items={len(items)}, B={result.bucket_count}, "
        f"Q={query_array.size}, resizes={result.resize_count}, "
        f"max_kicks={result.max_kicks_used}"
    )
    print("queries(first 16):", query_array[:16].tolist())
    print("found(first 16)  :", golden_found[:16].tolist())
    print("values(first 16) :", golden_values[:16].tolist())


if __name__ == "__main__":
    main()

In [ ]:
demo_dir = WORK_DIR / "build"
generator = WORK_DIR / "scripts" / "gen_data.py"
print(" demo CPU Golden")
subprocess.run(
    [sys.executable, str(generator), "--case", "demo"],
    cwd=demo_dir,
    check=True,
)

import numpy as np

# meta 记录桶数 B、查询数 Q 和插入的键值对数量。
bucket_count, query_count, item_count = map(
    int, (demo_dir / "meta.txt").read_text().split()
)

# table0/table1 是一维 packed table，reshape 后可理解为 [bucket, slot, key/value]。
table0 = np.fromfile(demo_dir / "input/table0.bin", dtype=np.int32)
table1 = np.fromfile(demo_dir / "input/table1.bin", dtype=np.int32)
queries = np.fromfile(demo_dir / "input/queries.bin", dtype=np.int32)

# Golden 是 CPU 双桶查找结果，后续 NPU 输出必须逐元素完全一致。
golden_values = np.fromfile(
    demo_dir / "output/golden_values.bin", dtype=np.int32
)
golden_found = np.fromfile(
    demo_dir / "output/golden_found.bin", dtype=np.int32
)
print(f"B={bucket_count}, Q={query_count}, items={item_count}")
print("table0 shape:", table0.reshape(bucket_count, 4, 2).shape)
print("queries      :", queries)
print("golden value :", golden_values)
print("golden found :", golden_found)
assert table0.size == table1.size == bucket_count * 8

## 4. 对比链式访问与固定两桶访问

链地址法在冲突桶后挂链表，query 命中或未命中的访问次数取决于链长；连续桶化数组则让每个 query 固定读取两个 32B 桶并检查 8 个候选槽。下面用一个小模型直接对比三类访问：

1. 链地址法：沿 `next` 指针逐节点查找，访问次数随链长变化；
2. 线性探测：在连续数组中顺序探测，冲突集中时可能形成长探测段；
3. 桶化 Cuckoo Hash：每个 query 固定访问两个连续桶，候选数量有稳定上界。

In [ ]:
def linked_chain_probe_count(chain_lengths, bucket_id):
    # 链地址法体现：桶里挂链表，query 需要沿链逐节点比较。
    # 链长越长，读取节点次数越多；不同 query 的循环次数可能完全不同。
    return chain_lengths[bucket_id] + (chain_lengths[bucket_id] == 0)


def linear_probe_count(occupied, start):
    # 开放寻址/线性探测体现：冲突后在同一连续数组中继续向后找。
    # 它仍然是连续数组访问，但冲突集中时探测步数会变长。
    count = 0
    index = start
    while occupied[index]:
        count += 1
        index = (index + 1) % len(occupied)
        if count == len(occupied):
            break
    # 若遇到空槽，最后一次访问空槽也要计入查询成本。
    return count + (count < len(occupied))


# 模拟 6 个桶的链长：bucket 3 的长链代表冲突集中场景。
chain_lengths = [1, 4, 0, 7, 2, 5]
linked_counts = [
    linked_chain_probe_count(chain_lengths, bucket_id)
    for bucket_id in range(len(chain_lengths))
]

# 模拟线性探测：前 12 个槽已占用，后 4 个槽为空。
occupied = [True] * 12 + [False] * 4
linear_counts = [linear_probe_count(occupied, start) for start in range(16)]

# 连续桶化 Cuckoo 查询体现：两个哈希函数给出两个候选桶，每桶 4 个连续槽。
# 它仍有随机桶地址，但每个桶内部是固定 32B 连续访问，候选槽数也固定。
cuckoo_bucket_loads = 2
cuckoo_candidate_slots = 2 * 4
cuckoo_table_bytes = 2 * 32

print("链地址法节点访问次数:", linked_counts)
print("线性探测访问次数    :", linear_counts)
print("Cuckoo 固定桶数     :", cuckoo_bucket_loads)
print("Cuckoo 固定槽数     :", cuckoo_candidate_slots)
print("Cuckoo 表访问字节   :", f"{cuckoo_table_bytes}B/query")
assert max(linked_counts) > cuckoo_bucket_loads
assert max(linear_counts) > cuckoo_bucket_loads

## 5. 设计多核 Tiling 与 UB

Host 使用 `blockNum=min(Q, availableVectorCores)`，每核获得连续 query 区间；核内按 `QUERY_TILE=256` 处理。这里体现连续数组查找的并行优势：query、value 和 found 都是连续数组，每个核负责一段连续区间，输出位置只有一个写入者。

UB 中 query、value 和 found 各采用深度 2 的队列，两个候选桶各占 32B。该预算远低于 DAV_2201 的 192KB，但“占用小”不等于“已经完成性能优化”；随机 GM 访问仍可能是主要瓶颈。

In [ ]:
QUERY_TILE = 256

# UB 预算体现连续数组分块：query/value/found 都按 QUERY_TILE 成段搬运。
# 两个候选桶是哈希查找带来的随机位置，但桶内部仍是 32B 连续数组。
ub_bytes = {
    "query double buffer": 2 * QUERY_TILE * 4,
    "value double buffer": 2 * QUERY_TILE * 4,
    "found double buffer": 2 * QUERY_TILE * 4,
    "table0 bucket": 32,
    "table1 bucket": 32,
}
print("单核主要 UB 预算")
for name, size in ub_bytes.items():
    print(f"{name:22s}: {size:5d} B")
total = sum(ub_bytes.values())
print(f"total: {total} B ({total / 1024:.2f} KiB)")
assert total < 192 * 1024

## 6. 编写 Tiling 结构

Tiling 头文件只使用标准 C++ 类型，由 Host 与 Kernel 共享。`queriesPerCore=ceil(Q/blockNum)`，尾核通过 `min` 计算实际 query 数。

In [ ]:
%%writefile $WRITE_ROOT/op_kernel/cuckoo_hash_lookup_tiling.h
#pragma once

#include <cstdint>

// 每个桶固定 4 个槽，每个槽保存 key/value 两个 int32。
// 这正是“连续桶化数组”的最小访问单元：4 * 2 * 4B = 32B。
constexpr uint32_t BUCKET_SIZE = 4;
constexpr uint32_t ENTRY_WORDS = 2;
constexpr uint32_t BUCKET_WORDS = BUCKET_SIZE * ENTRY_WORDS;

// 每个核一次处理的连续 query 数量，用于 UB 分块和尾块处理。
constexpr uint32_t QUERY_TILE = 256;

// Host 计算这些参数，Kernel 只读取，不在设备侧重新推导分核策略。
struct CuckooHashLookupTilingData {
    uint32_t blockNum;        // 实际启用核数。
    uint32_t bucketCount;     // 每张连续桶化表的桶数，必须为二次幂。
    uint32_t queryCount;      // query 总数。
    uint32_t queriesPerCore;  // 每个核负责的连续 query 区间上限。
};

## 7. 编写 Ascend C Kernel

Kernel 分为 `Init`、`Process`、`CopyInQueries`、`LoadBuckets`、`Compute` 和 `CopyOut`：

1. `Init` 计算当前核的 query 区间并分配 UB；
2. query 和输出按 256 个分块；
3. 两个哈希函数使用无符号 32 位溢出，与 Python 保持一致；
4. 每个候选桶用一次对齐 `DataCopy` 搬入；
5. 在 UB 中扫描 8 个 key；
6. value/found 使用 `DataCopyPad` 连续写回。

GM 不使用逐元素 `GetValue/SetValue`；逐元素检查只发生在 32B 的 UB 桶内。

In [ ]:
%%writefile $WRITE_ROOT/op_kernel/cuckoo_hash_lookup_kernel.cpp
#include "kernel_operator.h"
#include "cuckoo_hash_lookup_tiling.h"

// 连续数组查找 Kernel：每次启动完成整批 query。
// 与链地址法不同，这里不追逐 next 指针；每个 query 固定检查两个连续桶。
class KernelCuckooHashLookup {
public:
    __aicore__ inline KernelCuckooHashLookup(AscendC::TPipe *pipe) : pipe_(pipe) {}

    __aicore__ inline void Init(GM_ADDR table0, GM_ADDR table1, GM_ADDR queries,
                                GM_ADDR valuesOut, GM_ADDR foundOut,
                                const __gm__ CuckooHashLookupTilingData *tiling)
    {
        tiling_ = tiling;
        // Tiling 体现：每个核只处理自己负责的一段连续 query。
        const uint32_t blockIdx = AscendC::GetBlockIdx();
        startQuery_ = blockIdx * tiling_->queriesPerCore;
        const uint32_t remaining = startQuery_ < tiling_->queryCount
            ? tiling_->queryCount - startQuery_ : 0;
        ownedQueries_ = remaining < tiling_->queriesPerCore
            ? remaining : tiling_->queriesPerCore;

        // 将 GM 参数绑定为有类型的 GlobalTensor，桶数组长度单位为 int32。
        const uint32_t tableWords = tiling_->bucketCount * BUCKET_WORDS;
        table0Gm_.SetGlobalBuffer(
            reinterpret_cast<__gm__ int32_t *>(table0), tableWords);
        table1Gm_.SetGlobalBuffer(
            reinterpret_cast<__gm__ int32_t *>(table1), tableWords);
        queriesGm_.SetGlobalBuffer(
            reinterpret_cast<__gm__ int32_t *>(queries), tiling_->queryCount);
        valuesOutGm_.SetGlobalBuffer(
            reinterpret_cast<__gm__ int32_t *>(valuesOut), tiling_->queryCount);
        foundOutGm_.SetGlobalBuffer(
            reinterpret_cast<__gm__ int32_t *>(foundOut), tiling_->queryCount);

        // 查询与输出采用双缓冲；两个候选桶使用独立的 32B 输入队列。
        pipe_->InitBuffer(queryQueue_, 2, QUERY_TILE * sizeof(int32_t));
        pipe_->InitBuffer(valueQueue_, 2, QUERY_TILE * sizeof(int32_t));
        pipe_->InitBuffer(foundQueue_, 2, QUERY_TILE * sizeof(int32_t));
        pipe_->InitBuffer(bucket0Queue_, 1, BUCKET_WORDS * sizeof(int32_t));
        pipe_->InitBuffer(bucket1Queue_, 1, BUCKET_WORDS * sizeof(int32_t));
    }

    // Process 是典型 CopyIn -> Compute -> CopyOut 流程的调度入口。
    __aicore__ inline void Process()
    {
        if (ownedQueries_ == 0) {
            return;
        }
        // ownedQueries_ 是当前核实际负责的 query 数，尾核可能少于 queriesPerCore。
        for (uint32_t localBase = 0; localBase < ownedQueries_;
             localBase += QUERY_TILE) {
            const uint32_t remaining = ownedQueries_ - localBase;
            const uint32_t count = remaining < QUERY_TILE ? remaining : QUERY_TILE;
            CopyInQueries(localBase, count);
            Compute(localBase, count);
            CopyOut(localBase, count);
        }
    }

private:
    __aicore__ inline uint32_t Mix32(uint32_t value) const
    {
        // 哈希混合函数必须与 Python/Host 保持一致，保证三端桶号相同。
        value ^= value >> 16;
        value *= 0x7FEB352DU;
        value ^= value >> 15;
        value *= 0x846CA68BU;
        value ^= value >> 16;
        return value;
    }

    __aicore__ inline uint32_t Hash1(uint32_t key) const
    {
        // 第一个候选桶：查询 table0。
        return Mix32(key ^ 0x243F6A88U) & (tiling_->bucketCount - 1);
    }

    __aicore__ inline uint32_t Hash2(uint32_t key) const
    {
        // 第二个候选桶：查询 table1。
        return Mix32(key ^ 0x9E3779B9U) & (tiling_->bucketCount - 1);
    }

    __aicore__ inline void CopyInQueries(uint32_t localBase, uint32_t count)
    {
        // query 是连续数组，因此可以按 count 个 int32 一次搬入 UB。
        AscendC::LocalTensor<int32_t> queryLocal =
            queryQueue_.AllocTensor<int32_t>();
        AscendC::DataCopyExtParams params{
            1, static_cast<uint32_t>(count * sizeof(int32_t)), 0, 0, 0};
        AscendC::DataCopyPadExtParams<int32_t> pad{false, 0, 0, 0};
        AscendC::DataCopyPad(
            queryLocal, queriesGm_[startQuery_ + localBase], params, pad);
        queryQueue_.EnQue(queryLocal);
    }

    // LoadBuckets 是连续桶化查找的关键：哈希得到随机桶号，但一次搬运整桶连续数据。
    __aicore__ inline void LoadBuckets(
        uint32_t key,
        AscendC::LocalTensor<int32_t> &bucket0Local,
        AscendC::LocalTensor<int32_t> &bucket1Local)
    {
        // bucket0Local/bucket1Local 分别保存两个候选桶的 8 个 int32。
        bucket0Local = bucket0Queue_.AllocTensor<int32_t>();
        bucket1Local = bucket1Queue_.AllocTensor<int32_t>();
        const uint32_t offset0 = Hash1(key) * BUCKET_WORDS;
        const uint32_t offset1 = Hash2(key) * BUCKET_WORDS;

        // 哈希只决定桶地址；桶内部是连续数组。
        // 每个桶恰好为 8 个 int32，即一次对齐的 32B 连续搬运。
        AscendC::DataCopy(bucket0Local, table0Gm_[offset0], BUCKET_WORDS);
        AscendC::DataCopy(bucket1Local, table1Gm_[offset1], BUCKET_WORDS);
        bucket0Queue_.EnQue(bucket0Local);
        bucket1Queue_.EnQue(bucket1Local);
        bucket0Local = bucket0Queue_.DeQue<int32_t>();
        bucket1Local = bucket1Queue_.DeQue<int32_t>();
    }

    __aicore__ inline void Compute(uint32_t localBase, uint32_t count)
    {
        AscendC::LocalTensor<int32_t> queryLocal =
            queryQueue_.DeQue<int32_t>();
        // valueLocal/foundLocal 在 UB 中暂存本 tile 的输出，最后连续写回 GM。
        AscendC::LocalTensor<int32_t> valueLocal =
            valueQueue_.AllocTensor<int32_t>();
        AscendC::LocalTensor<int32_t> foundLocal =
            foundQueue_.AllocTensor<int32_t>();

        for (uint32_t i = 0; i < count; ++i) {
            const uint32_t key =
                static_cast<uint32_t>(queryLocal.GetValue(i));
            AscendC::LocalTensor<int32_t> bucket0Local;
            AscendC::LocalTensor<int32_t> bucket1Local;
            LoadBuckets(key, bucket0Local, bucket1Local);

            int32_t resultValue = 0;
            int32_t resultFound = 0;
            // 即使第一张表命中，也继续检查第二个桶，使每个 query 流程固定。
            // 扫描 table0 的 4 个连续槽；槽内偶数位置是 key，奇数位置是 value。
            for (uint32_t slot = 0; slot < BUCKET_SIZE; ++slot) {
                const uint32_t offset = slot * ENTRY_WORDS;
                if (bucket0Local.GetValue(offset) == static_cast<int32_t>(key)) {
                    resultValue = bucket0Local.GetValue(offset + 1);
                    resultFound = 1;
                }
            }
            // 再扫描 table1 的 4 个连续槽；即使 table0 命中也保持固定流程。
            for (uint32_t slot = 0; slot < BUCKET_SIZE; ++slot) {
                const uint32_t offset = slot * ENTRY_WORDS;
                if (bucket1Local.GetValue(offset) == static_cast<int32_t>(key)) {
                    resultValue = bucket1Local.GetValue(offset + 1);
                    resultFound = 1;
                }
            }
            valueLocal.SetValue(i, resultValue);
            foundLocal.SetValue(i, resultFound);

            // 释放两个候选桶的 UB Tensor，下一 query 可复用这两块 32B 空间。
            bucket0Queue_.FreeTensor(bucket0Local);
            bucket1Queue_.FreeTensor(bucket1Local);
        }

        queryQueue_.FreeTensor(queryLocal);
        valueQueue_.EnQue(valueLocal);
        foundQueue_.EnQue(foundLocal);
    }

    __aicore__ inline void CopyOut(uint32_t localBase, uint32_t count)
    {
        AscendC::LocalTensor<int32_t> valueLocal =
            valueQueue_.DeQue<int32_t>();
        AscendC::LocalTensor<int32_t> foundLocal =
            foundQueue_.DeQue<int32_t>();
        AscendC::DataCopyExtParams params{
            1, static_cast<uint32_t>(count * sizeof(int32_t)), 0, 0, 0};
        const uint32_t globalOffset = startQuery_ + localBase;
        // value/found 输出也是连续数组，按 query 区间连续写回 GM。
        AscendC::DataCopyPad(valuesOutGm_[globalOffset], valueLocal, params);
        AscendC::DataCopyPad(foundOutGm_[globalOffset], foundLocal, params);
        valueQueue_.FreeTensor(valueLocal);
        foundQueue_.FreeTensor(foundLocal);
    }

    AscendC::TPipe *pipe_;
    const __gm__ CuckooHashLookupTilingData *tiling_;
    AscendC::GlobalTensor<int32_t> table0Gm_, table1Gm_, queriesGm_;
    AscendC::GlobalTensor<int32_t> valuesOutGm_, foundOutGm_;
    AscendC::TQue<AscendC::TPosition::VECIN, 2> queryQueue_;
    AscendC::TQue<AscendC::TPosition::VECIN, 1> bucket0Queue_, bucket1Queue_;
    AscendC::TQue<AscendC::TPosition::VECOUT, 2> valueQueue_, foundQueue_;
    uint32_t startQuery_ = 0;
    uint32_t ownedQueries_ = 0;
};

// ascendc_library 根据该入口生成 Host 使用的 ACLRT Kernel 启动头文件。
// Kernel 入口参数顺序必须与 Host 侧 ACLRT_LAUNCH_KERNEL 调用保持一致。
extern "C" __global__ __aicore__ void cuckoo_hash_lookup_kernel(
    GM_ADDR table0, GM_ADDR table1, GM_ADDR queries,
    GM_ADDR valuesOut, GM_ADDR foundOut, GM_ADDR tiling)
{
    AscendC::TPipe pipe;
    KernelCuckooHashLookup op(&pipe);
    op.Init(table0, table1, queries, valuesOut, foundOut,
            reinterpret_cast<__gm__ CuckooHashLookupTilingData *>(tiling));
    op.Process();
}

## 8. 编写 C++ Host

`data_utils.h` 负责严格按元素数量读取和写入二进制文件。Host 主程序负责验证 packed table、查询 Vector Core 数、生成 Tiling、分配 GM、启动一次 Kernel、同步 stream 并写出结果。

这里的 Host 也承担教学边界检查：它确认输入确实是连续桶化数组，确认 key 位于对应哈希函数计算出的桶中，并拒绝负 query、非二次幂桶数和损坏空槽。

In [ ]:
%%writefile $WRITE_ROOT/op_host/data_utils.h
#pragma once

#include <fstream>
#include <stdexcept>
#include <string>
#include <vector>

template <typename T>
// 按元素个数读取二进制文件，避免文本解析影响实验主题。
std::vector<T> ReadBinary(const std::string &path, size_t count)
{
    std::ifstream file(path, std::ios::binary | std::ios::ate);
    if (!file) {
        throw std::runtime_error("cannot open " + path);
    }
    const auto bytes = static_cast<size_t>(file.tellg());
    if (bytes != count * sizeof(T)) {
        throw std::runtime_error("unexpected byte size for " + path);
    }
    file.seekg(0);
    std::vector<T> data(count);
    file.read(reinterpret_cast<char *>(data.data()),
              static_cast<std::streamsize>(bytes));
    return data;
}

template <typename T>
// 输出也写成二进制，验证脚本可以直接用 numpy.fromfile 精确读取。
void WriteBinary(const std::string &path, const std::vector<T> &data)
{
    std::ofstream file(path, std::ios::binary | std::ios::trunc);
    if (!file) {
        throw std::runtime_error("cannot create " + path);
    }
    file.write(reinterpret_cast<const char *>(data.data()),
               static_cast<std::streamsize>(data.size() * sizeof(T)));
}

In [ ]:
%%writefile $WRITE_ROOT/op_host/cuckoo_hash_lookup_main.cpp
#include <algorithm>
#include <cstdint>
#include <iostream>
#include <stdexcept>
#include <string>
#include <unordered_set>
#include <vector>

#include "acl/acl.h"
#include "aclrtlaunch_cuckoo_hash_lookup_kernel.h"
#include "data_utils.h"
#include "../op_kernel/cuckoo_hash_lookup_tiling.h"

// ACL_CHECK 把 ACL 返回码统一转成异常，便于学生定位失败步骤。
#define ACL_CHECK(call) do { \
    const aclError ret = (call); \
    if (ret != ACL_SUCCESS) { \
        throw std::runtime_error(std::string(#call) + " failed: " + \
                                 std::to_string(ret)); \
    } \
} while (0)

// Host 侧哈希函数与 Python、Kernel 保持完全一致。
uint32_t Mix32(uint32_t value)
{
    value ^= value >> 16;
    value *= 0x7FEB352DU;
    value ^= value >> 15;
    value *= 0x846CA68BU;
    value ^= value >> 16;
    return value;
}

uint32_t Hash1(uint32_t key, uint32_t bucketCount)
{
    return Mix32(key ^ 0x243F6A88U) & (bucketCount - 1);
}

uint32_t Hash2(uint32_t key, uint32_t bucketCount)
{
    return Mix32(key ^ 0x9E3779B9U) & (bucketCount - 1);
}

// 连续数组输入一次性从 Host 搬到 GM，避免逐元素设备侧初始化。
template <typename T>
uint8_t *MallocAndCopy(const std::vector<T> &host)
{
    uint8_t *device = nullptr;
    const size_t bytes = host.size() * sizeof(T);
    ACL_CHECK(aclrtMalloc(reinterpret_cast<void **>(&device), bytes,
                         ACL_MEM_MALLOC_HUGE_FIRST));
    ACL_CHECK(aclrtMemcpy(device, bytes, host.data(), bytes,
                         ACL_MEMCPY_HOST_TO_DEVICE));
    return device;
}

// 校验 packed table：确认它是“连续桶化数组”，不是链表或损坏布局。
void ValidateTable(const std::vector<int32_t> &table0,
                   const std::vector<int32_t> &table1,
                   const std::vector<int32_t> &queries,
                   uint32_t bucketCount)
{
    if (bucketCount == 0 || (bucketCount & (bucketCount - 1)) != 0) {
        throw std::invalid_argument("bucketCount must be a positive power of two");
    }
    const size_t expectedWords = static_cast<size_t>(bucketCount) * BUCKET_WORDS;
    if (table0.size() != expectedWords || table1.size() != expectedWords) {
        throw std::invalid_argument("packed table length mismatch");
    }
    std::unordered_set<int32_t> seen;
    for (uint32_t tableId = 0; tableId < 2; ++tableId) {
        const auto &table = tableId == 0 ? table0 : table1;
        for (uint32_t bucket = 0; bucket < bucketCount; ++bucket) {
            for (uint32_t slot = 0; slot < BUCKET_SIZE; ++slot) {
                const size_t offset =
                    static_cast<size_t>(bucket) * BUCKET_WORDS +
                    slot * ENTRY_WORDS;
                const int32_t key = table[offset];
                const int32_t value = table[offset + 1];
                if (key == -1) {
                    if (value != 0) {
                        throw std::invalid_argument(
                            "empty slots must store value 0");
                    }
                    continue;
                }
                if (key < 0 || !seen.insert(key).second) {
                    throw std::invalid_argument("invalid or duplicate key");
                }
                // 哈希体现：key 必须位于 hash1/hash2 指定的候选桶中。
                const uint32_t expectedBucket = tableId == 0
                    ? Hash1(static_cast<uint32_t>(key), bucketCount)
                    : Hash2(static_cast<uint32_t>(key), bucketCount);
                if (bucket != expectedBucket) {
                    throw std::invalid_argument("key is in an invalid bucket");
                }
            }
        }
    }
    if (queries.empty()) {
        throw std::invalid_argument("queryCount must be positive");
    }
    for (int32_t key : queries) {
        if (key < 0) {
            throw std::invalid_argument("queries must be nonnegative");
        }
    }
}

int main(int argc, char **argv)
{
    if (argc != 3) {
        std::cerr << "usage: cuckoo_hash_lookup <bucket_count> <query_count>\n";
        return 2;
    }
    const uint32_t bucketCount =
        static_cast<uint32_t>(std::stoul(argv[1]));
    const uint32_t queryCount =
        static_cast<uint32_t>(std::stoul(argv[2]));

    // Host 主流程：读输入 -> 校验布局 -> 申请设备内存 -> 启动 Kernel -> 取回输出。
    try {
        const size_t tableWords =
            static_cast<size_t>(bucketCount) * BUCKET_WORDS;
        std::cout << "[Host 1/5] Reading packed tables and queries\n";
        // 读取连续桶化输入；table0/table1 长度都应为 bucketCount * 8。
        const auto table0 =
            ReadBinary<int32_t>("input/table0.bin", tableWords);
        const auto table1 =
            ReadBinary<int32_t>("input/table1.bin", tableWords);
        const auto queries =
            ReadBinary<int32_t>("input/queries.bin", queryCount);
        ValidateTable(table0, table1, queries, bucketCount);
        std::cout << "           Validated B=" << bucketCount
                  << ", Q=" << queryCount << "\n";

        std::cout << "[Host 2/5] Initializing ACL device 0 and stream\n";
        ACL_CHECK(aclInit(nullptr));
        ACL_CHECK(aclrtSetDevice(0));
        aclrtStream stream = nullptr;
        ACL_CHECK(aclrtCreateStream(&stream));

        int64_t availableCores = 0;
        ACL_CHECK(aclrtGetDeviceInfo(
            0, ACL_DEV_ATTR_VECTOR_CORE_NUM, &availableCores));
        if (availableCores <= 0) {
            throw std::runtime_error("device reports no available Vector Core");
        }
        const uint32_t blockNum = std::min<uint32_t>(
            queryCount, static_cast<uint32_t>(availableCores));
        CuckooHashLookupTilingData tiling{
            blockNum,
            bucketCount,
            queryCount,
            (queryCount + blockNum - 1) / blockNum,
        };
        std::cout << "[Host 3/5] Tiling: blocks=" << blockNum
                  << ", queries/core=" << tiling.queriesPerCore << "\n";

        std::cout << "[Host 4/5] Copying GM buffers and launching Kernel once\n";
        uint8_t *table0Device = MallocAndCopy(table0);
        uint8_t *table1Device = MallocAndCopy(table1);
        uint8_t *queriesDevice = MallocAndCopy(queries);
        uint8_t *valuesDevice = nullptr;
        uint8_t *foundDevice = nullptr;
        uint8_t *tilingDevice = nullptr;
        ACL_CHECK(aclrtMalloc(
            reinterpret_cast<void **>(&valuesDevice),
            queryCount * sizeof(int32_t), ACL_MEM_MALLOC_HUGE_FIRST));
        ACL_CHECK(aclrtMalloc(
            reinterpret_cast<void **>(&foundDevice),
            queryCount * sizeof(int32_t), ACL_MEM_MALLOC_HUGE_FIRST));
        ACL_CHECK(aclrtMalloc(
            reinterpret_cast<void **>(&tilingDevice),
            sizeof(tiling), ACL_MEM_MALLOC_HUGE_FIRST));
        ACL_CHECK(aclrtMemcpy(
            tilingDevice, sizeof(tiling), &tiling, sizeof(tiling),
            ACL_MEMCPY_HOST_TO_DEVICE));

        // 启动一次 Kernel 完成所有 query 的连续桶化查找。
        ACLRT_LAUNCH_KERNEL(cuckoo_hash_lookup_kernel)(
            blockNum, stream, table0Device, table1Device, queriesDevice,
            valuesDevice, foundDevice, tilingDevice);
        ACL_CHECK(aclrtSynchronizeStream(stream));

        // Kernel 完成后把连续输出从 GM 拷回 Host，再写入 output 目录。
        std::vector<int32_t> values(queryCount);
        std::vector<int32_t> found(queryCount);
        ACL_CHECK(aclrtMemcpy(
            values.data(), values.size() * sizeof(int32_t), valuesDevice,
            values.size() * sizeof(int32_t), ACL_MEMCPY_DEVICE_TO_HOST));
        ACL_CHECK(aclrtMemcpy(
            found.data(), found.size() * sizeof(int32_t), foundDevice,
            found.size() * sizeof(int32_t), ACL_MEMCPY_DEVICE_TO_HOST));
        WriteBinary("output/values.bin", values);
        WriteBinary("output/found.bin", found);
        std::cout << "[Host 5/5] Wrote output/values.bin and output/found.bin\n";

        aclrtFree(tilingDevice);
        aclrtFree(foundDevice);
        aclrtFree(valuesDevice);
        aclrtFree(queriesDevice);
        aclrtFree(table1Device);
        aclrtFree(table0Device);
        aclrtDestroyStream(stream);
        aclrtResetDevice(0);
        aclFinalize();
        std::cout << "CuckooHashLookup completed successfully.\n";
        return 0;
    } catch (const std::exception &error) {
        std::cerr << "ERROR: " << error.what() << "\n";
        return 1;
    }
}

## 9. 编写精确验证和非法输入测试

输出为整数，因此采用逐元素完全相等，不使用浮点容差。验证脚本同时比较 value 与 found；非法输入脚本覆盖负 key、重复 key、负 query、非二次幂桶数、长度错误和损坏空槽。

这部分体现“教学实验要可验证”：连续桶化查找的结果必须和 CPU Golden 完全一致，输入布局不满足要求时必须尽早报错。

In [ ]:
%%writefile $WRITE_ROOT/scripts/verify_result.py
from pathlib import Path

import numpy as np


def main():
    # 精确验证：NPU 输出必须与 CPU Golden 完全一致。
    _, query_count, _ = map(int, Path("meta.txt").read_text().split())
    values = np.fromfile("output/values.bin", dtype=np.int32)
    found = np.fromfile("output/found.bin", dtype=np.int32)
    golden_values = np.fromfile("output/golden_values.bin", dtype=np.int32)
    golden_found = np.fromfile("output/golden_found.bin", dtype=np.int32)
    arrays = (values, found, golden_values, golden_found)
    if any(array.size != query_count for array in arrays):
        raise SystemExit("FAILED: output or golden size mismatch")
    # value 和 found 都必须一致；只比较 value 会漏掉未命中语义错误。
    values_ok = np.array_equal(values, golden_values)
    found_ok = np.array_equal(found, golden_found)
    print(f"values_exact={values_ok}, found_exact={found_ok}, Q={query_count}")
    if not values_ok or not found_ok:
        mismatch = np.flatnonzero((values != golden_values) | (found != golden_found))
        print("first mismatch indices:", mismatch[:16])
        print("values:", values[mismatch[:16]])
        print("golden:", golden_values[mismatch[:16]])
        print("found:", found[mismatch[:16]])
        print("golden_found:", golden_found[mismatch[:16]])
        raise SystemExit("FAILED")
    print("PASSED")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile $WRITE_ROOT/scripts/validate_inputs.py
import numpy as np

from cuckoo_utils import (
    BUCKET_WORDS,
    build_cuckoo,
    lookup_cpu,
    validate_packed_tables,
)


def expect_error(label, function):
    # 非法输入必须被拒绝，否则教学中的边界条件就不清晰。
    try:
        function()
    except ValueError as error:
        print(f"PASSED {label}: {error}")
    else:
        raise AssertionError(f"{label} was not rejected")


def main():
    expect_error("negative key", lambda: build_cuckoo([(-1, 2)]))
    expect_error("duplicate key", lambda: build_cuckoo([(1, 2), (1, 3)]))
    # 先生成一份合法 packed table，再故意破坏它来验证校验逻辑。
    result = build_cuckoo([(key, key * 10) for key in range(12)])
    expect_error(
        "negative query",
        lambda: lookup_cpu(result.table0, result.table1, result.bucket_count, [-2]),
    )
    expect_error(
        "non-power-of-two buckets",
        lambda: validate_packed_tables(
            np.zeros(3 * BUCKET_WORDS, np.int32),
            np.zeros(3 * BUCKET_WORDS, np.int32),
            3,
        ),
    )
    expect_error(
        "wrong packed length",
        lambda: validate_packed_tables(
            result.table0[:-1], result.table1, result.bucket_count
        ),
    )
    # 损坏空槽：EMPTY_KEY 槽的 value 必须为 0。
    corrupted = result.table0.copy()
    occupied = np.flatnonzero(corrupted[0::2] >= 0)
    if occupied.size:
        corrupted[int(occupied[0]) * 2] = -1
        corrupted[int(occupied[0]) * 2 + 1] = 123
    expect_error(
        "invalid empty slot",
        lambda: validate_packed_tables(
            corrupted, result.table1, result.bucket_count
        ),
    )
    # 移错桶：key 不在 hash1/hash2 指定桶中时必须报错。
    displaced = result.table0.copy().reshape(
        result.bucket_count, 4, 2
    )
    occupied = np.argwhere(displaced[:, :, 0] >= 0)
    if occupied.size and result.bucket_count > 1:
        bucket, slot = map(int, occupied[0])
        wrong_bucket = (bucket + 1) % result.bucket_count
        displaced[wrong_bucket, 0] = displaced[bucket, slot]
        displaced[bucket, slot] = (-1, 0)
        expect_error(
            "key in wrong bucket",
            lambda: validate_packed_tables(
                displaced.reshape(-1), result.table1, result.bucket_count
            ),
        )
    print("All invalid-input checks PASSED")


if __name__ == "__main__":
    main()

In [ ]:
print("执行 CPU 非法输入测试")
# 该测试不依赖 NPU，用来验证 CPU 工具对边界条件的拒绝是否清晰。
subprocess.run(
    [sys.executable, str(WORK_DIR / "scripts/validate_inputs.py")],
    cwd=WORK_DIR,
    check=True,
)

## 10. 配置 CMake 与一键脚本

`ascendc_library` 编译带 `.cpp` 后缀的 Ascend C Kernel，并生成 `aclrtlaunch_cuckoo_hash_lookup_kernel.h`；普通 C++ 编译器构建 Host，再链接 Kernel 目标与 CANN Runtime。`run.sh` 干净构建一次后运行全部案例。

In [ ]:
%%writefile $WRITE_ROOT/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)
project(cuckoo_hash_lookup LANGUAGES CXX)

set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)
set(CMAKE_CXX_EXTENSIONS OFF)

if(NOT CMAKE_BUILD_TYPE)
  set(CMAKE_BUILD_TYPE Release CACHE STRING "Build type" FORCE)
endif()

# SOC_VERSION 对应目标 NPU 芯片；实验默认使用 Atlas 910B3/A2。
set(SOC_VERSION "ascend910b3" CACHE STRING "Ascend SoC version")
set(RUN_MODE "npu" CACHE STRING "Ascend C run mode")
set(ASCEND_CANN_PATH "$ENV{ASCEND_HOME_PATH}" CACHE PATH "CANN installation path")
if(NOT ASCEND_CANN_PATH)
  set(ASCEND_CANN_PATH "$ENV{ASCEND_TOOLKIT_HOME}"
      CACHE PATH "CANN installation path" FORCE)
endif()
if(NOT ASCEND_CANN_PATH)
  message(FATAL_ERROR "ASCEND_HOME_PATH or ASCEND_TOOLKIT_HOME is not set")
endif()

set(ASCEND_CANN_PACKAGE_PATH "${ASCEND_CANN_PATH}"
    CACHE PATH "CANN package path" FORCE)
set(CMAKE_INSTALL_PREFIX "${CMAKE_BINARY_DIR}/out"
    CACHE PATH "Ascend C output path" FORCE)

# 不同 CANN 镜像的 ascendc.cmake 位置可能不同，因此列出多个候选路径。
set(ASCENDC_CMAKE_CANDIDATES
  "${ASCEND_CANN_PACKAGE_PATH}/tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake"
  "${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake"
  "${ASCEND_CANN_PACKAGE_PATH}/aarch64-linux/tikcpp/ascendc_kernel_cmake/ascendc.cmake"
  "${ASCEND_CANN_PACKAGE_PATH}/x86_64-linux/tikcpp/ascendc_kernel_cmake/ascendc.cmake"
)
foreach(candidate IN LISTS ASCENDC_CMAKE_CANDIDATES)
  if(EXISTS "${candidate}")
    set(ASCENDC_CMAKE_FILE "${candidate}")
    break()
  endif()
endforeach()
if(NOT ASCENDC_CMAKE_FILE)
  message(FATAL_ERROR "Cannot find ascendc.cmake under ${ASCEND_CANN_PACKAGE_PATH}")
endif()

message(STATUS "ASCEND_CANN_PACKAGE_PATH=${ASCEND_CANN_PACKAGE_PATH}")
message(STATUS "SOC_VERSION=${SOC_VERSION}")
include("${ASCENDC_CMAKE_FILE}")

# 编译 Ascend C Kernel，并生成对应的 ACLRT 启动头文件。
ascendc_library(cuckoo_hash_lookup_kernels STATIC
  op_kernel/cuckoo_hash_lookup_kernel.cpp
)
ascendc_include_directories(cuckoo_hash_lookup_kernels PRIVATE
  ${CMAKE_CURRENT_SOURCE_DIR}/op_kernel
)
ascendc_compile_definitions(cuckoo_hash_lookup_kernels PRIVATE
  -DASCENDC_DUMP=0
)

# 编译普通 C++ Host，并链接 Kernel 目标与 CANN 运行库。
add_executable(cuckoo_hash_lookup
  op_host/cuckoo_hash_lookup_main.cpp
)
# Host 需要同时包含自己头文件、Kernel 生成头文件和 CANN Runtime 头文件。
target_include_directories(cuckoo_hash_lookup PRIVATE
  op_host
  op_kernel
  ${ASCEND_CANN_PACKAGE_PATH}/include
  ${ASCEND_CANN_PACKAGE_PATH}/include/external
  ${ASCEND_CANN_PACKAGE_PATH}/runtime/include
  ${CMAKE_INSTALL_PREFIX}/include/cuckoo_hash_lookup_kernels
  ${CMAKE_BINARY_DIR}/out/include/cuckoo_hash_lookup_kernels
)
target_link_directories(cuckoo_hash_lookup PRIVATE
  ${ASCEND_CANN_PACKAGE_PATH}/lib64
  ${ASCEND_CANN_PACKAGE_PATH}/runtime/lib64/stub
  ${ASCEND_CANN_PACKAGE_PATH}/runtime/lib64
  ${ASCEND_CANN_PACKAGE_PATH}/acllib/lib64
  ${ASCEND_CANN_PACKAGE_PATH}/aarch64-linux/devlib
  ${ASCEND_CANN_PACKAGE_PATH}/x86_64-linux/devlib
)
target_compile_definitions(cuckoo_hash_lookup PRIVATE
  SOC_VERSION="${SOC_VERSION}"
)
# 链接 Kernel 静态库和 ACL/CANN 运行时库，形成可执行直调程序。
target_link_libraries(cuckoo_hash_lookup PRIVATE
  cuckoo_hash_lookup_kernels
  ascendcl
  tiling_api
  register
  platform
  ascendalog
  c_sec
  dl
)
add_dependencies(cuckoo_hash_lookup cuckoo_hash_lookup_kernels)

In [ ]:
%%writefile $WRITE_ROOT/run.sh
#!/usr/bin/env bash
set -euo pipefail

SCRIPT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"
cd "${SCRIPT_DIR}"
# 缺少 CANN 环境时直接失败，避免后续 CMake 报错信息过长。
: "${ASCEND_HOME_PATH:?ASCEND_HOME_PATH is not set}"
source "${ASCEND_HOME_PATH}/set_env.sh"

# 从干净目录构建，避免旧 CMake 缓存和生成头文件影响结果。
rm -rf build
mkdir -p build
cd build
cmake .. -DSOC_VERSION="${SOC_VERSION:-ascend910b3}"
make -j4

# 回归案例覆盖命中、未命中、碰撞、尾块和随机装载，确保教学边界都跑到。
cases=(demo empty single hits misses mixed duplicate_queries negative_values
       collision tail255 tail256 tail257 random_low random_high)
for case_name in "${cases[@]}"; do
    echo "=== case: ${case_name} ==="
    # 每个 case 重新生成输入和 Golden，避免复用上一个案例的输出。
    python3 ../scripts/gen_data.py --case "${case_name}"
    read -r bucket_count query_count item_count < meta.txt
    rm -f output/values.bin output/found.bin
    ./cuckoo_hash_lookup "${bucket_count}" "${query_count}"
    test -f output/values.bin
    test -f output/found.bin
    python3 ../scripts/verify_result.py
done
python3 ../scripts/validate_inputs.py
echo "All CuckooHashLookup cases PASSED"

In [ ]:
%%writefile $WRITE_ROOT/README.md
# CuckooHashLookup 直调实验工程

本工程由 `06.02_memory_access_compare.ipynb` 的 `%%writefile` 单元生成，展示适合 NPU 的连续桶化数组查找实现。
链地址法不在 Ascend C 中实现，只作为访存不规则性的教学对照；上板部分聚焦连续数组布局如何降低实现复杂度。

- `scripts/cuckoo_utils.py`：CPU 建表、输入校验和 Golden 查询。
- `scripts/gen_data.py`：生成两张扁平表、query 和精确参考结果。
- `op_kernel/`：Tiling 结构与 Ascend C Kernel。
- `op_host/`：ACL 初始化、设备内存管理、Kernel 启动与结果写回。
- `scripts/verify_result.py`：对 value 和 found 执行整数精确比较。
- `run.sh`：构建并运行全部合法及非法输入案例。

目标环境：CANN 9.0.0、Atlas 910B3/A2、ARM64、`ascend910b3`。

In [ ]:
print("查看生成的工程结构")
# 只展示文件相对路径，帮助学生对应 notebook 中各个 %%writefile 单元。
for path in sorted(WORK_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(WORK_DIR))

## 11. 编译工程

构建单元先删除旧 `build`，避免生成头文件和缓存残留。完整日志保存为 `build/build.log`；失败时显示最后 80 行。无 NPU 环境时跳过，不影响前面的 CPU 教学。

In [ ]:
if NPU_READY:
    build_dir = WORK_DIR / "build"
    shutil.rmtree(build_dir, ignore_errors=True)
    build_dir.mkdir(parents=True, exist_ok=True)
    command = (
        'source "$ASCEND_HOME_PATH/set_env.sh" && '
        'cmake .. -DSOC_VERSION=ascend910b3 && '
        'cmake --build . --clean-first -j4'
    )
    print("编译 Kernel 与 Host")
    result = subprocess.run(
        ["bash", "-lc", command],
        cwd=build_dir,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    log_path = build_dir / "build.log"
    log_path.write_text(result.stdout, encoding="utf-8")
    lines = result.stdout.splitlines()
    print("\n".join(lines[-80:] if result.returncode else lines[-25:]))
    if result.returncode:
        raise RuntimeError(f"编译失败，完整日志：{log_path}")
    print("可执行程序：", build_dir / "cuckoo_hash_lookup")
else:
    print("跳过 NPU 编译：请在 CANNLab 910B3 环境重跑本单元。")

## 12. 交互运行一个案例

修改 `CASE_NAME` 可以观察不同输入。流程为：生成表和 Golden → 读取 meta → 启动 Host → 读取 NPU 输出 → 精确比较。

In [ ]:
CASE_NAME = "demo"  # 可改为 collision、tail257、random_high 等

if NPU_READY:
    build_dir = WORK_DIR / "build"
    print(f"[运行 1/4] 生成 {CASE_NAME} 输入")
    subprocess.run(
        [sys.executable, str(WORK_DIR / "scripts/gen_data.py"),
         "--case", CASE_NAME],
        cwd=build_dir,
        check=True,
    )
    bucket_count, query_count, item_count = map(
        int, (build_dir / "meta.txt").read_text().split()
    )
    print(f"B={bucket_count}, Q={query_count}, items={item_count}")

    print("[运行 2/4] 调用 C++ Host 并启动一次 NPU Kernel")
    # Host 可执行程序会启动一次 NPU Kernel，完成全部 query。
    run_command = (
        'source "$ASCEND_HOME_PATH/set_env.sh" && '
        f'./cuckoo_hash_lookup {bucket_count} {query_count}'
    )
    subprocess.run(["bash", "-lc", run_command], cwd=build_dir, check=True)

    print("[运行 3/4] 展示 NPU 输出")
    print("values:", np.fromfile(
        build_dir / "output/values.bin", dtype=np.int32
    )[:32])
    print("found :", np.fromfile(
        build_dir / "output/found.bin", dtype=np.int32
    )[:32])

    print("[运行 4/4] 与 CPU Golden 精确比较")
    subprocess.run(
        [sys.executable, str(WORK_DIR / "scripts/verify_result.py")],
        cwd=build_dir,
        check=True,
    )
else:
    print("跳过 NPU 单案例；CPU 建表与 Golden 已完成。")

## 13. 完整回归

`run.sh` 覆盖空表、单元素、全命中、全未命中、混合、重复 query、负 value、碰撞、255/256/257 尾块以及两种随机装载。每个案例独立重新生成输入和 Golden。

In [ ]:
# 完整回归会重新构建并运行多个 case，耗时比单案例更长。
if NPU_READY:
    print("[完整回归] 调用：", WORK_DIR / "run.sh")
    subprocess.run(["bash", "run.sh"], cwd=WORK_DIR, check=True)
else:
    print("跳过 NPU 完整回归；请在 CANNLab 910B3 环境执行。")

## 14. 实验总结

本实验把不确定的冲突处理留在 CPU 建表阶段，把 NPU 查询收敛为固定两桶的连续数组访问。它展示了链表与连续数组之间的关键差异：链表更灵活，但访问路径依赖数据；连续数组更适合分块搬运、对齐访问和多核均分。



## 课后练习

1. `Q=1000`、可用 Vector Core 为 20 时，`blockNum` 和 `queriesPerCore` 分别是多少？
2. 链地址法为什么会让不同 query 的循环次数不同？
3. 连续桶化方案中，每个 query 的表访问量和候选比较次数是多少？
4. 为什么 value 必须和 found 分开输出？

In [ ]:
!cat answer/06.02_memory_access_compare/answers.md